# TDW6323 — Malaysian Cost of Living Analysis

**Thesis:** In which Malaysian states is life getting less affordable — where are prices rising faster than income — and which spending categories are driving it?

---

## Step 0 — Schema Check

Confirm real column names, shapes, and dtypes before writing any pipeline code.

In [1]:
import pandas as pd
import os

DATA_DIR = os.path.join("..", "data", "parquet")
DATASETS = ["hh_income", "hies_state", "cpi_2d_state"]

In [2]:
# --- hh_income ---
hh_income = pd.read_parquet(os.path.join(DATA_DIR, "hh_income.parquet"))
print("Shape:", hh_income.shape)
print("Columns:", hh_income.columns.tolist())
print("\nDtypes:")
print(hh_income.dtypes)
print("\nNull counts:")
print(hh_income.isnull().sum())
hh_income.head()

Shape: (21, 3)
Columns: ['date', 'income_mean', 'income_median']

Dtypes:
date             object
income_mean       Int64
income_median     Int64
dtype: object

Null counts:
date             0
income_mean      0
income_median    0
dtype: int64


,date,income_mean,income_median
0,1970-01-01,264,166
1,1974-01-01,362,227
2,1976-01-01,505,308
3,1979-01-01,678,429
4,1984-01-01,1098,719


In [3]:
# --- hies_state ---
hies_state = pd.read_parquet(os.path.join(DATA_DIR, "hies_state.parquet"))
print("Shape:", hies_state.shape)
print("Columns:", hies_state.columns.tolist())
print("\nDtypes:")
print(hies_state.dtypes)
print("\nNull counts:")
print(hies_state.isnull().sum())
hies_state.head()

Shape: (16, 7)
Columns: ['date', 'state', 'income_mean', 'income_median', 'expenditure_mean', 'gini', 'poverty']

Dtypes:
date                 object
state                   str
income_mean           int64
income_median         int64
expenditure_mean      int64
gini                float64
poverty             float64
dtype: object

Null counts:
date                0
state               0
income_mean         0
income_median       0
expenditure_mean    0
gini                0
poverty             0
dtype: int64


,date,state,income_mean,income_median,expenditure_mean,gini,poverty
0,2022-01-01,Johor,8517,6879,5342,0.36646,4.6
1,2022-01-01,Kedah,5550,4402,3765,0.35938,9.0
2,2022-01-01,Kelantan,4885,3614,3505,0.38540,13.2
3,2022-01-01,Melaka,8057,6210,5707,0.36963,4.2
4,2022-01-01,Negeri Sembilan,6788,5226,4678,0.36853,4.4


In [4]:
# --- cpi_2d_state ---
cpi = pd.read_parquet(os.path.join(DATA_DIR, "cpi_2d_state.parquet"))
print("Shape:", cpi.shape)
print("Columns:", cpi.columns.tolist())
print("\nDtypes:")
print(cpi.dtypes)
print("\nNull counts:")
print(cpi.isnull().sum())
cpi.head()

Shape: (44128, 4)
Columns: ['state', 'date', 'division', 'index']

Dtypes:
state                  str
date        datetime64[ns]
division               str
index              float64
dtype: object

Null counts:
state       0
date        0
division    0
index       0
dtype: int64


,state,date,division,index
0,Johor,2010-01-01,overall,99.4
1,Johor,2010-02-01,overall,99.4
2,Johor,2010-03-01,overall,99.4
3,Johor,2010-04-01,overall,99.4
4,Johor,2010-05-01,overall,99.6


In [5]:
# Unique state strings in each dataset — check for inconsistencies before merging
print("=== hh_income states ===")
print(sorted(hh_income["state"].unique()) if "state" in hh_income.columns else "no 'state' column")

print("\n=== hies_state states ===")
print(sorted(hies_state["state"].unique()) if "state" in hies_state.columns else "no 'state' column")

print("\n=== cpi_2d_state states ===")
print(sorted(cpi["state"].unique()) if "state" in cpi.columns else "no 'state' column")

=== hh_income states ===
no 'state' column

=== hies_state states ===
['Johor', 'Kedah', 'Kelantan', 'Melaka', 'Negeri Sembilan', 'Pahang', 'Perak', 'Perlis', 'Pulau Pinang', 'Sabah', 'Sarawak', 'Selangor', 'Terengganu', 'W.P. Kuala Lumpur', 'W.P. Labuan', 'W.P. Putrajaya']

=== cpi_2d_state states ===
['Johor', 'Kedah', 'Kelantan', 'Melaka', 'Negeri Sembilan', 'Pahang', 'Perak', 'Perlis', 'Pulau Pinang', 'Sabah', 'Sarawak', 'Selangor', 'Terengganu', 'W.P. Kuala Lumpur', 'W.P. Labuan', 'W.P. Putrajaya']


In [6]:
# Date ranges for each dataset
for name, df in [("hh_income", hh_income), ("hies_state", hies_state), ("cpi_2d_state", cpi)]:
    if "date" in df.columns:
        d = pd.to_datetime(df["date"])
        print(f"{name}: {d.min()} → {d.max()}")
    else:
        print(f"{name}: no 'date' column — columns are {df.columns.tolist()}")

hh_income: 1970-01-01 00:00:00 → 2022-01-01 00:00:00
hies_state: 2022-01-01 00:00:00 → 2022-01-01 00:00:00
cpi_2d_state: 2010-01-01 00:00:00 → 2026-05-01 00:00:00


In [7]:
# CPI: unique categories / division codes
for col in ["division", "group", "category", "series", "item"]:
    if col in cpi.columns:
        print(f"Unique '{col}' values ({cpi[col].nunique()}):")
        print(sorted(cpi[col].unique()))
        print()

Unique 'division' values (14):
['01', '02', '03', '04', '05', '06', '07', '08', '09', '10', '11', '12', '13', 'overall']

